
# Update since Wednesday 4 Nov 2026

A plain-words summary of the fall-detection work: what we did, how, what came out, and where everything is.
No code here, just the story. All numbers come from the notebooks and scripts listed at the end.

**Short version:** we checked three fall datasets and picked **FallAllD** to train on. We built a fall model that runs **on the ESP32 itself**, tested it on a **different dataset (WEDA-FALL)**, then tuned it on the real prototype until hitting the table or putting the device down stopped causing false alarms. It now works well on the prototype.



## 1. The goal

SilverGuard is a wrist-worn **ESP32-C3 + MPU-6050** (accelerometer + gyroscope) at **50 readings per second**. It has to:
- say what the wearer is doing (**still / moving**), and
- notice a **fall** and raise an alert.

**Most important rule: missing a real fall is worse than a false alarm.** But too many false alarms make the device useless, so both are measured.



## 2. Step 1: Checking the three datasets

Before training anything, we loaded each dataset and checked its size, missing values, cleanliness and relevance to our device.

| | WEDA-FALL | UMAFall | FallAllD |
|---|---|---|---|
| Sensor position | wrist (Fitbit watch) | wrist + 4 other places | waist / wrist / neck |
| Speed | ~50 Hz, but irregular | wrist only ~20 Hz (too slow) | 238 Hz (we slow it to 50 Hz) |
| Recordings | 969 | 746 | 6,605 |
| Missing values | none | 86,233 (corrupted files) | none |
| Older adults falling? | no (older people did daily activities only) | no | no |

**What we found:**
- **FallAllD is the cleanest and biggest** (35 kinds of falls, 44 daily activities). It became our training set, using the wrist device only (device "D2").
- **WEDA-FALL is the closest match to our device** (wrist, ~50 Hz). It became our *test* set from "different hardware". Its problems: readings cut off at 4 g, and timestamps arrive in bursts, so we re-spaced them evenly at 50 Hz.
- **UMAFall had 20 files damaged by Excel** (decimal points turned into thousands separators). A corrected zip in the folder fixes them, but its wrist sensor is too slow for us, so we didn't use it further.
- **No dataset has an older person falling**, which is a limitation for the report.

Notebooks: `choosing dataset.ipynb`, `datasets explained.ipynb`



## 3. How we set up every experiment (the approach)

The same recipe was used every time, so results are comparable:

1. **Same units as our device:** accelerometer in **g**, gyroscope in **deg/s**, **50 Hz**.
2. **Cut recordings into 2.5-second windows**, sliding forward ~1.2 s each time.
3. **Labels:** a window that contains the **impact** of a fall (the biggest spike) = **fall**. Everything else = **not fall**.
   - Why not label the whole recording? In FallAllD every fall sits in the middle of the recording, so a model could cheat by learning "spike in the middle = fall".
4. **Split by person, never at random.** 3 people (subjects 1, 11, 12) were kept completely hidden as the final test. If the same person is in training and testing, the score looks great but is fake ("leakage").
5. **Turn each window into numbers ("features"):** for example the average, spread, min and max of the acceleration size, how sudden the change is (jerk), and whether the wrist is still afterwards.
6. **Judge with the right scores, not accuracy.** A model that always says "no fall" would be 97% accurate and useless. We use:
   - **Recall**: of the real falls, how many were caught? (most important)
   - **False alarms per hour** of normal activity
   - **Fall recordings caught**: did it notice each fall at least once?



## 4. Step 2: The first fall model (FallAllD, wrist)

We compared a simple rule with two machine-learning models on the 3 hidden people:

| Model | Recall | False alarms / hour | Falls caught |
|---|---|---|---|
| Rule: spike > 2.5 g | 100% | ~313 | 73/73 |
| Rule: spike > 3 g | 93% | ~214 | 68/73 |
| Logistic regression | 99% | ~48 | 73/73 |
| **Random forest** | **97%** | **~30** | **73/73** |

**Lesson:** the simple spike rule catches everything but cries wolf constantly (jogging and hopping also spike). Learning from data cut false alarms about **10x** while still catching every fall.

Notebook: `modeltrainingFallAllD.ipynb`



## 5. Step 3: The gyroscope

**What unit is it in?** FallAllD doesn't say, so we tested it with physics. When the wrist turns from one still position to another, the accelerometer shows how far gravity turned, and adding up the gyro readings must give the same angle.
- FallAllD gyro: **0.070 deg/s per count**. The error was 5° for the right scale vs 58-64° for the wrong ones.
- WEDA gyro: **rad/s** (later, same test).

**Your idea: "fall only if accel AND gyro both say fall."**
- Falls do spin the wrist hard (~470 deg/s), but so do the things that fool the accelerometer (hitting, swinging, hopping: ~200-520 deg/s).
- Result: **recall stayed 100%** (no falls lost), but false alarms only dropped from ~313 to **~275 per hour** (about 12% fewer).
- **Lesson:** gyro helps a bit, best as one input among many, not as a yes/no gate.

Notebook: `fallalld_gyro_eda_and_rule.ipynb`



## 6. Step 4: Combining both, with vs without gyro, and moving to another dataset

**Two-stage model** (this is what runs on the device now):
1. **Stage 1:** is there a spike above 2.5 g? If not, it's not a fall (cheap, catches 100% of falls).
2. **Stage 2:** a random forest looks at that spike and decides whether it was really a fall.

**With vs without gyro** (FallAllD, 3 hidden people):

| Features used | Recall | False alarms / hour |
|---|---|---|
| accel only, size of movement | 95% | ~85 |
| accel + gyro, size of movement | 95% | ~77 |
| accel only, every axis | 97% | ~39 |
| accel + gyro, every axis | 97% | ~32 |

Gyro removes about 10-17% of false alarms. "Every axis" works better on FallAllD, but it depends on how the watch is worn.

**Moving to another dataset (train on FallAllD, test on WEDA-FALL):** this shows how the model behaves on hardware it has never seen, like our prototype.
- **Falls still caught: 322-340 of 350** recordings, depending on the model.
- **False alarms jumped 4-8x** (about 134-265 per hour), because it is a different watch, different people, and different activities. This is called **domain shift**.
- **Size-of-movement features moved best** to the new watch (340/350 falls), because they don't care how the watch is rotated. So that is the version we put on the device (14 features).

**Lesson:** numbers measured on one dataset do not carry over to new hardware. Testing on the real prototype matters most.

Notebook: `two_stage_model_and_weda_test.ipynb`



## 7. Step 5: Putting it on the ESP32

1. **Raw logger first:** the board streams raw sensor data at 50 Hz; `record.py` saves labelled trials on the laptop. Settings are written straight to the sensor registers: ±16 g, ±2000 deg/s, 21 Hz filter, 50 Hz.
2. **Models exported to C** (with the `emlearn` tool):
   - **Activity model (still / moving):** the 5-feature UCI model.
     - We found a **bug in the old test sketch**: its model file expected 24 numbers but was only given 5, so its answers were garbage.
   - **Fall model:** shrunk to **30 trees** to fit the chip: 94% recall, ~86 false alarms/hour, still 73/73 falls on the hidden people (the 300-tree version: 95%, ~76).
3. **Self-test at boot:** 3 real windows with Python's answers are built in. The board checks its own maths matches Python and prints PASS/FAIL. We also checked on the laptop that the C version gives **identical** answers to Python on all 5,370 spike windows.
4. **Readable output:** `Activity: Still / Moving` every 0.5 s, and on a fall the board stops and repeats **"ALERT ... FALL DETECTEDDD!!!"** until you type `r`.

Uses about 30% of the chip's storage and 6% of its memory.

Folder: `firmware/silverguard_logger/`, made by `notebooks/fall model/export_models_to_c.py`



## 8. Step 6: Tuning on the real prototype

**Problem 1: hitting the table set off a fall alert.**
- This was not a bug: on WEDA's watch data the same model also alarmed on **54 of 73 "hit table" recordings**. A hard hit and a fall impact look alike in 2.5 s.
- **Fix: check what happens after the spike.** A real fall leaves the wrist in a new position (median turn 80°); a table hit barely moves it (median 21°). Now the board waits 2.5 s, and if the wrist turned less than **20°**, the spike is rejected.
- The 20° was chosen on FallAllD and checked on WEDA:

| Recordings with a fall alarm (WEDA) | Before | After |
|---|---|---|
| Hit table | 54/73 | 28/73 |
| Clapping | 63/74 | 39/74 |
| Older adults' daily activities | 49/157 | 26/157 |
| **Real falls** | **342/350** | **338/350** |

**Problem 2: just putting the device down on the table set off an alert.**
- Putting it down = a knock + a new resting position, so it passed every check.
- **Fix: check the 1 second before the knock.** In a real fall the wrist **drops** (acceleration dips well below 1 g) and **spins fast**. Setting it down by hand does neither. The spike is rejected if both are missing (stayed above 0.85 g **and** turned slower than 150 deg/s).
- Cost: **only 1 of 809** real falls in FallAllD + WEDA would be lost.
- **Result on the prototype: works well.**

Script with the evidence: `notebooks/fall model/confirm_stage_eval.py`



## 9. How the final system decides (every 0.5 s)

```
new 2.5 s of wrist data
   |
   +--> Activity model -> "Activity: Still / Moving"
   |
   +--> Stage 1: spike above 2.5 g?            no  -> not a fall
              | yes
        Stage 2: forest says fall?             no  -> not a fall
              | yes  ("possible fall, checking...")
        Stage 3a: wrist turned > 20 deg after? no  -> "NOT a fall: probably a hit / clap"
              | yes
        Stage 3b: dropped or spun before?      no  -> "NOT a fall: probably put down"
              | yes
        ALERT ALERT ... FALL DETECTEDDD!!!  (stops until 'r' or reset)
```
Each check removes a different kind of false alarm while losing almost no real falls.



## 10. Limits (put these in the report)

- **No older adults fall in any dataset.** All falls were by younger volunteers onto mattresses.
- **Trained on other people's hardware** (FallAllD logger, UCI phone at the waist). Our own prototype recordings are the real test.
- **Few test people:** only 3 hidden people in FallAllD, so numbers can shift with a different split.
- **Not fully confirmed:** that FallAllD's device "D2" is the wrist (our gyro check supports it, but it's not proof).
- **The prototype tuning values are starting points:** 20°, 0.85 g and 150 deg/s were set from data plus a few real tests.
- **This is a prototype, not a certified fall detector.**

## 11. Next steps
1. Record our own trials with `record.py` (staged falls onto a mattress, table hits, putting it down, walking, sitting) and test on them.
2. Retrain using our own recordings, so the model learns our hardware.
3. Count false alarms per *event* instead of per window, for a fairer number.
4. Ask the supervisor about ethics approval before any trials with older adults.



## 12. Where everything is

| What | File |
|---|---|
| Dataset check | `notebooks/fall model/dataset_audit_wedafall_umafall_fallalld.ipynb` |
| Datasets explained simply | `notebooks/fall model/datasets_explained_simply.ipynb` |
| First fall model | `notebooks/fall model/fallalld_wrist_fall_classifier.ipynb` |
| Gyro unit + accel AND gyro rule | `notebooks/fall model/fallalld_gyro_eda_and_rule.ipynb` |
| Two-stage, with/without gyro, WEDA test | `notebooks/fall model/two_stage_model_and_weda_test.ipynb` |
| Shared loading / features code | `notebooks/fall model/fad_utils.py` |
| Export models to C | `notebooks/fall model/export_models_to_c.py` |
| Evidence for the after-spike checks | `notebooks/fall model/confirm_stage_eval.py` |
| Device code (sketch, models, recorder, how-to) | `firmware/silverguard_logger/` (`silverguard_logger.ino`, `record.py`, `README.md`) |
